# MVP2 FDTD 128x128 -> 2D contour -> HDMI (4-lane)
**128x128** grid (4x finer than 64x64) on the 4-lane FDTD + 2D contour heatmap. source_addr is now 14-bit. Use `mag_mode=2` (signed Ey).
AXI: `gpio_ctrl 0x41200000` | `gpio_status 0x41210000`.


In [ ]:
from pynq import Overlay, MMIO
import time
ol = Overlay('fdtd_hdmi.bit')
print('overlay loaded'); print(list(ol.ip_dict.keys()))


In [ ]:
CTRL   = MMIO(0x41200000, 0x10000)
STATUS = MMIO(0x41220000, 0x10000)
MOTION = MMIO(0x41210000, 0x10000)   # moving source velocity + speed throttle
GPIO_CH1, GPIO_CH2 = 0x0, 0x8
GRID = 128
def cell(x, y): return y*GRID + x
def q313(v):    return int(round(v*8192)) & 0xFFFF

def set_ctrl(phase_step, amplitude, source_addr, solver_enable, mag_mode,
             sample_req, free_run, height_ctl=2):
    """mag_mode: 0=|E|, 1=|S|, 2=raw signed Ey (smooth waves). height_ctl signed -16..15."""
    ch1 = (q313(amplitude) << 16) | q313(phase_step)
    ch2 = (((mag_mode >> 1) & 1) << 23) | ((height_ctl & 0x1F) << 18) | ((free_run & 1) << 17) \
        | ((sample_req & 1) << 16) | ((mag_mode & 1) << 15) | ((solver_enable & 1) << 14) \
        | (source_addr & 0x3FFF)
    CTRL.write(GPIO_CH1, ch1); CTRL.write(GPIO_CH2, ch2)

def set_amplitude(amp):
    """Set source amplitude (Q3.13 float, e.g. 0..0.5) live; keeps phase_step."""
    ch1 = (CTRL.read(GPIO_CH1) & 0x0000FFFF) | (q313(amp) << 16)
    CTRL.write(GPIO_CH1, ch1)

def set_height(h):
    ch2 = (CTRL.read(GPIO_CH2) & ~(0x1F<<18)) | ((h & 0x1F)<<18); CTRL.write(GPIO_CH2, ch2)

def clear_fields():
    """Reset the simulation: zero Ey/Ex/Bz (all 4 lanes). Edge-triggered bit 22."""
    v = CTRL.read(GPIO_CH2)
    CTRL.write(GPIO_CH2, v | (1<<24)); time.sleep(0.005); CTRL.write(GPIO_CH2, v & ~(1<<24))
    print("fields cleared")

def read_status():
    chk = STATUS.read(GPIO_CH1); s = STATUS.read(GPIO_CH2)
    return dict(checksum=chk, solver_done=(s>>0)&1, source_valid=(s>>1)&1, mag_done=(s>>2)&1,
                mag_busy=(s>>3)&1, source_latched=(s>>4)&1, pp_read_sel=(s>>5)&1,
                pp_frame_ready=(s>>6)&1, bridge_busy=(s>>7)&1, source_q313=(s>>16)&0xFFFF)
print("helpers ready")

def _s16(v):  # float cells/iter -> signed 16-bit (x256 fixed point)
    iv = int(round(v*256))
    iv = max(-32768, min(32767, iv))
    return iv & 0xFFFF

def set_velocity(vx, vy):
    """Move the source at (vx,vy) cells per FDTD iteration. The velocity is
       latched on a move_en rising edge, so we pulse move_en low->high here to
       reload it live (the source restarts from the source cell each call).
       Subsonic |v| -> Doppler (rings compress ahead); supersonic -> Mach cone.
       The numerical wave speed is < 0.5 cell/iter -- sweep v upward to find the
       regimes. Call stop_motion() to halt."""
    MOTION.write(0x0, (_s16(vy) << 16) | _s16(vx))
    base = MOTION.read(0x8) & ~0x1          # move_en low (re-arm the edge)
    MOTION.write(0x8, base)
    MOTION.write(0x8, base | 0x1)           # rising edge -> reload vx,vy + restart
    print(f"moving source: vx={vx} vy={vy} cells/iter (sweep v up: rings->Doppler->Mach cone)")

def stop_motion():
    MOTION.write(0x8, MOTION.read(0x8) & ~0x1)
    print("source motion stopped (back to static source_addr)")

def set_speed(idle_cycles):
    """Throttle: idle cycles inserted between FDTD iterations. 0 = full speed;
       larger = slower (watchable). e.g. set_speed(200000) ~ visibly slowed."""
    ch2 = (MOTION.read(0x8) & 0xFF) | ((idle_cycles & 0xFFFFFF) << 8)
    MOTION.write(0x8, ch2)
    print(f"speed throttle: {idle_cycles} idle cycles/iteration")

def set_source_mode(dcfree=True):
    """dcfree=True: inject the DC-free first-difference of the sine. A MOVING
       source otherwise deposits a net DC offset on each cell it passes, which
       freezes into a static 'trail'; the difference telescopes -> no trail.
       Stationary waves are unchanged. If waves look weak, raise amplitude or
       set_height(+1..+2) for display gain."""
    ch2 = MOTION.read(0x8)
    ch2 = (ch2 | 0x2) if dcfree else (ch2 & ~0x2)
    MOTION.write(0x8, ch2)
    print(f"source mode: {'DC-free (no trail)' if dcfree else 'direct sine'}")

def set_source_field(field='Bz'):
    """Choose which field the point source drives (motion CH2 bit2):
       'Ey' = dipole source -> figure-8 pattern with two nulls (real EM dipole);
       'Bz' = monopole source -> isotropic CIRCULAR ripples (best for Doppler/Mach,
              no quiet wedges). Switchable live."""
    bz = 1 if str(field).lower().startswith('b') else 0
    ch2 = MOTION.read(0x8)
    ch2 = (ch2 | 0x4) if bz else (ch2 & ~0x4)
    MOTION.write(0x8, ch2)
    print(f"source field: {'Bz (monopole, circular)' if bz else 'Ey (dipole, two nulls)'}")

## 5. UDP source-magnitude control (ESP32 -> PS)
The ESP32 streams the probe value over WiFi/UDP (port **5005**, ASCII int per packet); a background thread on the PS maps it to the FDTD **source amplitude** live. Both devices must be on the same local network. ESP32 sketch: `esp32/source_magnitude_udp.ino`.

First calibrate the probe range with `mc.calibrate()` (move the probe through its full range, note min/max), then set `RAW_MIN/RAW_MAX`.

In [ ]:
import socket, threading, time

class MagnitudeUDP:
    """Receive probe values over UDP and drive the FDTD source amplitude.
    Packet = one ASCII integer per datagram (e.g. b'2731')."""
    def __init__(self, port=5005, raw_min=0, raw_max=4095, amp_min=0.0, amp_max=0.5):
        self.port=port; self.raw_min=raw_min; self.raw_max=raw_max
        self.amp_min=amp_min; self.amp_max=amp_max
        self.sock=socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        self.sock.bind(('0.0.0.0', port)); self.sock.setblocking(False)
        self._run=False; self._t=None; self.last_raw=None; self.last_amp=None
    def _latest(self):
        data=None
        while True:
            try: data,_=self.sock.recvfrom(64)
            except BlockingIOError: break
        if data is None: return None
        try: return int(data.decode('ascii','ignore').strip().split(',')[0])
        except (ValueError, IndexError): return None
    def _to_amp(self, raw):
        f=(raw-self.raw_min)/max(1,(self.raw_max-self.raw_min)); f=min(1.0,max(0.0,f))
        return self.amp_min + f*(self.amp_max-self.amp_min)
    def _loop(self):
        while self._run:
            r=self._latest()
            if r is not None:
                self.last_raw=r; self.last_amp=self._to_amp(r); set_amplitude(self.last_amp)
            time.sleep(0.01)
    def start(self):
        if not self._run:
            self._run=True; self._t=threading.Thread(target=self._loop, daemon=True); self._t.start()
            print(f'UDP magnitude control live on :{self.port}')
    def stop(self):
        self._run=False; time.sleep(0.05); print('stopped')
    def calibrate(self, secs=8):
        """Move the probe through its full range; prints min/max raw seen."""
        lo,hi=1<<30,-(1<<30); t0=time.time()
        print('move the probe through its full range...')
        while time.time()-t0 < secs:
            r=self._latest()
            if r is not None: lo=min(lo,r); hi=max(hi,r)
            time.sleep(0.01)
        print(f'raw range: min={lo} max={hi}  -> set raw_min/raw_max to these')
        return lo,hi

# start it (source stays at the centre; its amplitude tracks the probe)
mc = MagnitudeUDP(port=5005, raw_min=0, raw_max=4095, amp_min=0.0, amp_max=0.5)
# mc.calibrate()          # run once, then set raw_min/raw_max from the printout
mc.start()
# mc.stop()               # to stop the background thread / re-run the cell


## 1. Start free-run (4-lane solver)


In [ ]:
# NOTE: phase now advances phase_step PER ITERATION (CORDIC re-timed to
# one sample/iter). ~0.18 rad => ~12-cell wavelength; bigger=shorter waves.
# source at y=48 = centre of a lane (seams at y=32/64/96); keeps rings
# symmetric. y=64 is ON a seam -> top/bottom look mismatched.
set_ctrl(phase_step=0.18, amplitude=0.3, source_addr=cell(64,48),
         solver_enable=1, mag_mode=2, sample_req=1, free_run=1, height_ctl=2)
print("4-lane FDTD running. Tune relief with set_height(n); reset with clear_fields().")


## Doppler / moving source
Move the source through the grid. With `mag_mode=2` you'll see wavefronts bunch up ahead (higher freq) and stretch behind (lower freq). Push the speed past the wave speed (~0.35 cells/iter) for a Mach cone. Throttle the sim first so it's watchable.

In [ ]:
set_source_mode(True)    # DC-free injection: no trail
set_source_field('Bz')   # monopole -> full circular wavefronts (no two-sided nulls)
set_speed(150000)        # slow the sim so the motion is watchable

# velocity now applies LIVE (no notebook restart needed). Sweep it to see the
# regimes -- the numerical wave speed is somewhere < 0.5 cell/iter:
set_velocity(0.08, 0.0)  #  slow  -> near-symmetric rings (little Doppler)
# set_velocity(0.15, 0.0)  # medium -> Doppler: rings bunch ahead, stretch behind
# set_velocity(0.30, 0.0)  # near/above wave speed -> fronts pile into a cone
# set_velocity(0.45, 0.0)  # clearly supersonic -> Mach cone / shock
# set_velocity(0.18, 0.10) # diagonal
# stop_motion()            # halt (source stays put)


## 2. Confirm alive
Checksum must keep changing and ping-pong must swap. The solver now produces frames ~4x faster than the single-lane build.


In [ ]:
seen=set(); flips=0; prev=None
for _ in range(20):
    st=read_status(); seen.add(st['checksum'])
    if prev is not None and st['pp_read_sel']!=prev: flips+=1
    prev=st['pp_read_sel']; time.sleep(0.05)
print("unique checksums:", len(seen), " read_sel flips:", flips)
assert len(seen)>1, "checksum frozen"
print("PASS: 4-lane solver live.")


## 3. Reset / clean the renderer


In [ ]:
clear_fields()   # zero the field, wave restarts from the source


## 4. HDMI
Expect the **same** 3D wave terrain as the single-lane build (try `mag_mode=2` signed-Ey). The win is internal: the solver advances 4x faster per iteration. Tuning: `set_height(1..4)`, `phase_step` for wavelength (now per-iteration: ~0.18 rad ≈ 12 cells; raise for shorter waves), `mag_mode` 0/1/2 for view.


## 6. Hardware panel input (ESP32 resistive touch panel -> PS over USB)

The ESP32 one-panel firmware streams a readable `DATA,...` line over USB serial at ~50 Hz. In **Mode 2** (stylus) a touch sets the **source position**; the amplitude pot sets source strength, the Clear button resets the field, and the 3D pots (yaw/pitch/zoom) drive the camera. The panel reports a **64x38** grid cell; `PanelReceiver` maps it to the 128x128 FDTD grid and writes `source_addr` live (`move_en=0`). **Mode 1** (conducting-sheet probe measurement) is captured in `.last` but does not move the source. Firmware: `esp32/fdtd_panel_input.ino`; port `/dev/ttyUSB0` or `/dev/ttyACM0`.

In [ ]:
import threading, time
try:
    import serial   # pyserial: `pip install pyserial` on the PYNQ if missing
except ImportError:
    serial = None; print('pyserial not installed: run  pip install pyserial')

# panel geometry reported by the ESP32 one-panel firmware
PANEL_X, PANEL_Y = 64, 38          # ESP32 sends grid cells x 0..63, y 0..37

def set_source_position(x, y):
    """Place the STATIC source at FDTD cell (x,y) live. move_en=0 so the PS owns
       the position (drag-a-stylus mode, distinct from the Doppler sweep)."""
    x = max(0, min(GRID-1, int(x))); y = max(0, min(GRID-1, int(y)))
    MOTION.write(0x8, MOTION.read(0x8) & ~0x1)            # Doppler auto-motion off
    CTRL.write(GPIO_CH2, (CTRL.read(GPIO_CH2) & ~0x3FFF) | (cell(x, y) & 0x3FFF))

def _panel_to_fdtd(gx, gy):
    """64x38 panel cell -> 128x128 FDTD cell (independent per-axis scale; the
       panel is landscape so y is stretched -- fine for 'source follows stylus')."""
    fx = round(gx * (GRID-1) / (PANEL_X-1))
    fy = round(gy * (GRID-1) / (PANEL_Y-1))
    return fx, fy

def parse_data_line(s):
    """One 'DATA,k=v,...' USB line from the firmware -> dict (ints where possible).
       Non-DATA lines (boot/calibration messages) return None."""
    if not s.startswith('DATA,'): return None
    d = {}
    for kv in s[5:].split(','):
        if '=' in kv:
            k, v = kv.split('=', 1)
            try: d[k] = int(v)
            except ValueError: d[k] = v
    return d

class PanelReceiver:
    """Reads the ESP32 one-panel firmware over USB serial (the ~50 Hz 'DATA,...'
       text lines, USB_PYNQ_TEXT=1) and drives the FDTD:
         Mode 2 + touch (not 'wall')  -> source POSITION  (drag the stylus)
         amplitude pot                -> source amplitude (deadbanded)
         clear button (rising edge)   -> clear_fields()
         3D pots yaw/pitch/zoom       -> set_camera()  [d4s48 build only]
       Mode 1 (conducting-sheet probe) is captured in .last but does NOT move
       the source. Port: /dev/ttyUSB0 or /dev/ttyACM0 (check `ls /dev/tty*`)."""
    def __init__(self, port='/dev/ttyUSB0', baud=115200,
                 do_position=True, do_amp=True, do_clear=True, do_camera=True):
        if serial is None: raise RuntimeError('pyserial not installed')
        self.ser = serial.Serial(port, baud, timeout=0)
        self.do_position = do_position
        self.do_amp      = do_amp
        self.do_clear    = do_clear
        self.do_camera   = do_camera and ('set_camera' in globals())  # d4s48 only
        self._run=False; self._t=None; self._buf=b''; self.last=None
        self._clear_prev=0; self._amp_prev=-99; self._cam_prev=(-99,-99,-99); self._cam_t=0.0
    def _lines(self):
        try: self._buf += self.ser.read(8192)
        except Exception: return []
        if b'\n' not in self._buf: return []
        parts = self._buf.split(b'\n'); self._buf = parts[-1]
        return [p.decode('ascii','ignore').strip() for p in parts[:-1]]
    def _apply(self, d):
        if self.do_clear and d.get('clear',0)==1 and self._clear_prev==0:
            clear_fields()                                   # rising edge only
        self._clear_prev = d.get('clear',0)
        if (self.do_position and d.get('mode')==2 and d.get('touch')==1
                and d.get('wall',0)==0 and 'x' in d and 'y' in d):
            fx, fy = _panel_to_fdtd(d['x'], d['y']); set_source_position(fx, fy)
        if self.do_amp and 'amp' in d and abs(d['amp']-self._amp_prev) > 8:
            self._amp_prev = d['amp']; set_amplitude(d['amp']/1023*0.5)
        if self.do_camera and all(k in d for k in ('yaw','pitch','zoom')):
            cur=(d['yaw'],d['pitch'],d['zoom']); now=time.time()
            if now-self._cam_t > 0.08 and any(abs(a-b)>8 for a,b in zip(cur,self._cam_prev)):
                self._cam_prev=cur; self._cam_t=now
                set_camera(yaw_deg=d['yaw']/1023*360,
                           pitch_deg=2+d['pitch']/1023*87,
                           dist=0.4+d['zoom']/1023*1.6)
    def _loop(self):
        while self._run:
            for ln in self._lines():
                rec = parse_data_line(ln)
                if rec: self.last = rec; self._apply(rec)
            time.sleep(0.005)
    def start(self):
        if not self._run:
            self._run=True
            self._t=threading.Thread(target=self._loop, daemon=True); self._t.start()
            feats = ['position'] + (['amp'] if self.do_amp else []) \
                  + (['clear'] if self.do_clear else []) + (['camera'] if self.do_camera else [])
            print(f'panel receiver live on {self.ser.port}  ({", ".join(feats)})')
    def stop(self): self._run=False; time.sleep(0.05); print('stopped')

# --- usage ---
# Manual test (no panel):  set_source_position(20, 100)
# pr = PanelReceiver(port='/dev/ttyUSB0')   # or '/dev/ttyACM0'
# pr.start()        # flip the ESP32 to Mode 2, drag the stylus -> the source follows
# print(pr.last)    # latest decoded frame; Mode-1 probe value is pr.last['probe']
# pr.stop()

